## p43

In [6]:
from PIL import Image
import numpy as np
from pathlib import Path

# hacking_dir = Path('/Users/stephen/Stephencwelch Dropbox/welch_labs/vla/hackin')
hacking_dir=Path('/home/stephen/robots')
output_dir = hacking_dir / 'p43_patchified'
output_dir.mkdir(exist_ok=True)

image_names = ['base_0_rgb', 'left_wrist_0_rgb', 'right_wrist_0_rgb']

total_height = 2.72
grid_n = 16
patch_size_manim = total_height / grid_n
gap_factor = 0.12
scale = 1 + gap_factor
patch_px = 14  # native patch size

# Render scale — each native patch pixel becomes this many output pixels
render_scale = 4  # try 4x, bump to 6 or 8 if still uneven
patch_out = patch_px * render_scale

px_per_unit = patch_out / patch_size_manim

xs = [(j - 8 + 0.5) * patch_size_manim * scale for j in range(16)]
ys = [-(i - 8 + 0.5) * patch_size_manim * scale for i in range(2, 14)]

x_min = min(xs) - patch_size_manim / 2
x_max = max(xs) + patch_size_manim / 2
y_max = max(ys) + patch_size_manim / 2

canvas_w = int(np.ceil((x_max - x_min) * px_per_unit))
canvas_h = int(np.ceil((max(ys) - min(ys) + patch_size_manim) * px_per_unit))

print(f'patch_out={patch_out}px, canvas={canvas_w}x{canvas_h}, px_per_unit={px_per_unit:.1f}')

for frame_idx in range(0, 300):
    for k, name in enumerate(image_names):
        patch_dir = hacking_dir / f'p35/{frame_idx}/{name}'
        if not patch_dir.exists():
            continue

        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

        for idx_i, i in enumerate(range(2, 14)):
            for j in range(16):
                cx = (j - 8 + 0.5) * patch_size_manim * scale
                cy = -(i - 8 + 0.5) * patch_size_manim * scale

                px_x = int(round((cx - x_min) * px_per_unit)) - patch_out // 2
                px_y = int(round((y_max - cy) * px_per_unit)) - patch_out // 2

                patch_path = patch_dir / f'patch_{i}_{j}.png'
                # Upscale patch with nearest-neighbor to keep crisp pixels
                patch = Image.open(patch_path).convert('RGB')
                patch = patch.resize((patch_out, patch_out), Image.NEAREST)
                patch = np.array(patch)

                py_end = min(px_y + patch_out, canvas_h)
                px_end = min(px_x + patch_out, canvas_w)
                ph = py_end - px_y
                pw = px_end - px_x
                if ph > 0 and pw > 0:
                    canvas[px_y:py_end, px_x:px_end] = patch[:ph, :pw]

        Image.fromarray(canvas).save(output_dir / f'{frame_idx}_{k}.png')

    if frame_idx % 50 == 0:
        print(f'Done frame {frame_idx}')

patch_out=56px, canvas=997x746, px_per_unit=329.4
Done frame 0
Done frame 50
Done frame 100
Done frame 150
Done frame 200
Done frame 250


In [2]:
# # Jupyter cell — export patchified images with manim-matching gaps
# from PIL import Image
# import numpy as np
# from pathlib import Path

# # hacking_dir = Path('/Users/stephen/Stephencwelch Dropbox/welch_labs/vla/hackin')
# hacking_dir=Path('/home/stephen/robots')
# output_dir = hacking_dir / 'p43_patchified'
# output_dir.mkdir(exist_ok=True)

# total_height = 2.72
# grid_n = 16
# patch_size_manim = total_height / grid_n  # 0.17 manim units
# gap_factor = 0.12
# patch_px = 14  # each image patch is 14×14 pixels

# px_per_unit = patch_px / patch_size_manim

# # Compute canvas bounds (rows 2–13, cols 0–15, after gap expansion)
# scale = 1 + gap_factor
# xs = [(j - 8 + 0.5) * patch_size_manim * scale for j in range(16)]
# ys = [-(i - 8 + 0.5) * patch_size_manim * scale for i in range(2, 14)]

# x_min = min(xs) - patch_size_manim / 2
# x_max = max(xs) + patch_size_manim / 2
# y_max = max(ys) + patch_size_manim / 2  # top in manim coords

# canvas_w = int(np.ceil((x_max - x_min) * px_per_unit))
# canvas_h = int(np.ceil((max(ys) - min(ys) + patch_size_manim) * px_per_unit))

# image_names = ['base_0_rgb', 'left_wrist_0_rgb', 'right_wrist_0_rgb']

In [3]:
# for frame_idx in range(0, 300):
#     for k, name in enumerate(image_names):
#         patch_dir = hacking_dir / f'p35/{frame_idx}/{name}'
#         if not patch_dir.exists():
#             continue

#         canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

#         for idx_i, i in enumerate(range(2, 14)):
#             for j in range(16):
#                 cx = (j - 8 + 0.5) * patch_size_manim * scale
#                 cy = -(i - 8 + 0.5) * patch_size_manim * scale

#                 # Manim y-up → image y-down
#                 px_x = int(round((cx - x_min) * px_per_unit)) - patch_px // 2
#                 px_y = int(round((y_max - cy) * px_per_unit)) - patch_px // 2

#                 patch_path = patch_dir / f'patch_{i}_{j}.png'
#                 patch = np.array(Image.open(patch_path).convert('RGB'))[:patch_px, :patch_px]

#                 # Clip to canvas bounds just in case
#                 py_end = min(px_y + patch_px, canvas_h)
#                 px_end = min(px_x + patch_px, canvas_w)
#                 ph = py_end - px_y
#                 pw = px_end - px_x
#                 if ph > 0 and pw > 0:
#                     canvas[px_y:py_end, px_x:px_end] = patch[:ph, :pw]

#         Image.fromarray(canvas).save(output_dir / f'{frame_idx}_{k}.png')

#     if frame_idx % 50 == 0:
#         print(f'Done frame {frame_idx}')

Done frame 0
Done frame 50
Done frame 100
Done frame 150
Done frame 200
Done frame 250
